# 04 -- Model Version Pinning Demo

Implements the Chapter 6 design: a deployment config that pins an **explicit, dated model-version
identifier** (never an "always latest" alias), an evaluation-result logger that stamps every logged
row with that pinned version, and a scheduled check that flags when the pinned version is nearing a
**mocked** vendor-announced deprecation date -- before it turns into a request-failure incident. This
notebook pins a single research/production-baseline pair for clarity; Chapter 6 argues the real
five-agent system needs this same pattern applied once per agent, since each agent's deployment config
can drift to a different version independently of the others.

Entirely offline: the "vendor deprecation calendar" below is a small, hand-built mock dictionary, not
a real API call to Anthropic or any other vendor.

In [1]:
from dataclasses import dataclass, field
from datetime import date, timedelta
import pandas as pd

pd.set_option("display.max_colwidth", 60)
print("Environment ready. Offline, deterministic, no network calls.")

Environment ready. Offline, deterministic, no network calls.


## 1. A deployment config that pins an explicit version

Chapter 6's core fix: never reference a model by an "always latest" alias. The deployment config
below pins a specific, dated identifier for both the research model (Claude) and the production
baseline (Azure OpenAI GPT), so a vendor-side model update can never silently change what a given
deployment configuration actually points at.

In [2]:
@dataclass(frozen=True)
class ModelDeploymentConfig:
    role: str                 # "research" or "production_baseline"
    vendor: str
    pinned_model_version: str  # explicit, dated -- NEVER an "always latest" alias
    pinned_at: date


RESEARCH_DEPLOYMENT = ModelDeploymentConfig(
    role="research",
    vendor="anthropic",
    pinned_model_version="claude-sonnet-5-20260315",
    pinned_at=date(2026, 3, 20),
)

PRODUCTION_BASELINE_DEPLOYMENT = ModelDeploymentConfig(
    role="production_baseline",
    vendor="azure_openai",
    pinned_model_version="gpt-4o-enterprise-2025-08-deployment",
    pinned_at=date(2025, 8, 12),
)

for cfg in (RESEARCH_DEPLOYMENT, PRODUCTION_BASELINE_DEPLOYMENT):
    print(f"{cfg.role:22s} | vendor={cfg.vendor:14s} | pinned_version={cfg.pinned_model_version}")

research               | vendor=anthropic      | pinned_version=claude-sonnet-5-20260315
production_baseline    | vendor=azure_openai   | pinned_version=gpt-4o-enterprise-2025-08-deployment


## 2. Evaluation logger: stamp every result with the pinned version

Pinning the config is necessary but not sufficient on its own -- chapter 6 is explicit that the
version identifier also has to be logged on every single evaluation result row, not just in the
deployment config, so a later audit can check whether a score shift lines up with a version change.

In [3]:
@dataclass
class EvaluationResultLog:
    rows: list = field(default_factory=list)

    def log_result(self, case_id, deployment_config, faithfulness_score, run_date):
        self.rows.append(
            {
                "case_id": case_id,
                "role": deployment_config.role,
                "vendor": deployment_config.vendor,
                "pinned_model_version": deployment_config.pinned_model_version,
                "faithfulness_score": faithfulness_score,
                "run_date": run_date,
            }
        )

    def to_dataframe(self):
        return pd.DataFrame(self.rows)


logger = EvaluationResultLog()

# Simulate a handful of evaluation runs across a few days, all against the SAME pinned config.
simulated_runs = [
    ("SYN-001", 0.92, date(2026, 3, 21)),
    ("SYN-002", 0.88, date(2026, 3, 21)),
    ("SYN-003", 0.95, date(2026, 3, 25)),
    ("SYN-004", 0.90, date(2026, 3, 28)),
    ("SYN-005", 0.93, date(2026, 4, 2)),
]

for case_id, score, run_date in simulated_runs:
    logger.log_result(case_id, RESEARCH_DEPLOYMENT, score, run_date)

results_df = logger.to_dataframe()
results_df

,case_id,role,vendor,pinned_model_version,faithfulness_score,run_date
0,SYN-001,research,anthropic,claude-sonnet-5-20260315,0.92,2026-03-21
1,SYN-002,research,anthropic,claude-sonnet-5-20260315,0.88,2026-03-21
2,SYN-003,research,anthropic,claude-sonnet-5-20260315,0.95,2026-03-25
3,SYN-004,research,anthropic,claude-sonnet-5-20260315,0.90,2026-03-28
4,SYN-005,research,anthropic,claude-sonnet-5-20260315,0.93,2026-04-02


In [4]:
# Confirm every logged row carries the SAME pinned version -- this is what makes the evaluation
# auditable: if a future row ever showed a different version string, that would be visible
# immediately, rather than silently blending two different models' scores into one average.
distinct_versions = results_df["pinned_model_version"].nunique()
assert distinct_versions == 1, "Expected a single pinned version across this evaluation batch."
print(f"All {len(results_df)} logged evaluation rows carry the same pinned version: "
      f"{results_df['pinned_model_version'].iloc[0]}")

All 5 logged evaluation rows carry the same pinned version: claude-sonnet-5-20260315


## 3. What a silent version change WOULD look like, if it happened

To make the failure mode chapter 6 warns about concrete: simulate what the log would contain if,
partway through the same evaluation window, the deployment had been left on an unpinned "always
latest" reference and the vendor shipped a silent update. This block is illustrative only -- it does
not affect `RESEARCH_DEPLOYMENT` itself, which stays pinned throughout this notebook.

In [5]:
UNPINNED_SIMULATION = ModelDeploymentConfig(
    role="research",
    vendor="anthropic",
    pinned_model_version="claude-sonnet-5-20260401",  # a NEWER version, simulating a silent update
    pinned_at=date(2026, 4, 1),
)

hypothetical_logger = EvaluationResultLog()
hypothetical_runs = [
    ("SYN-001", 0.92, RESEARCH_DEPLOYMENT, date(2026, 3, 21)),
    ("SYN-002", 0.88, RESEARCH_DEPLOYMENT, date(2026, 3, 21)),
    ("SYN-003", 0.79, UNPINNED_SIMULATION, date(2026, 4, 3)),  # silently a different model now
    ("SYN-004", 0.81, UNPINNED_SIMULATION, date(2026, 4, 5)),
]
for case_id, score, cfg, run_date in hypothetical_runs:
    hypothetical_logger.log_result(case_id, cfg, score, run_date)

hypothetical_df = hypothetical_logger.to_dataframe()
hypothetical_df

,case_id,role,vendor,pinned_model_version,faithfulness_score,run_date
0,SYN-001,research,anthropic,claude-sonnet-5-20260315,0.92,2026-03-21
1,SYN-002,research,anthropic,claude-sonnet-5-20260315,0.88,2026-03-21
2,SYN-003,research,anthropic,claude-sonnet-5-20260401,0.79,2026-04-03
3,SYN-004,research,anthropic,claude-sonnet-5-20260401,0.81,2026-04-05


In [6]:
hypothetical_versions = hypothetical_df["pinned_model_version"].nunique()
print(f"Distinct model versions present in this hypothetical log: {hypothetical_versions}")
if hypothetical_versions > 1:
    print("WARNING (illustrative): this evaluation batch silently mixes results from two "
          "different model versions. Averaging faithfulness_score across all rows here would "
          "produce a number that doesn't correspond to any single model -- exactly the research-"
          "validity gap chapter 6 warns about. Because every row carries an explicit version "
          "string, this is CATCHABLE by a simple groupby, rather than invisible.")

# The fix in practice: never silently average across versions -- segment by version first.
segmented = hypothetical_df.groupby("pinned_model_version")["faithfulness_score"].mean().round(3)
print()
print("Correct handling -- segment by version before comparing:")
print(segmented)

Distinct model versions present in this hypothetical log: 2
WARNING (illustrative): this evaluation batch silently mixes results from two different model versions. Averaging faithfulness_score across all rows here would produce a number that doesn't correspond to any single model -- exactly the research-validity gap chapter 6 warns about. Because every row carries an explicit version string, this is CATCHABLE by a simple groupby, rather than invisible.

Correct handling -- segment by version before comparing:
pinned_model_version
claude-sonnet-5-20260315    0.9
claude-sonnet-5-20260401    0.8
Name: faithfulness_score, dtype: float64


## 4. Scheduled deprecation check

A mocked vendor-announced deprecation calendar, plus a scheduled check that flags a pinned version
once it's within a configurable warning window of its announced retirement date -- proactively,
rather than discovering the deprecation only once requests against that version start failing.

In [7]:
# Mocked vendor deprecation calendar -- stands in for a real query against the vendor's model
# catalog / deprecation announcements API. Maps pinned version identifiers to their announced
# retirement date.
MOCK_VENDOR_DEPRECATION_CALENDAR = {
    "claude-sonnet-5-20260315": date(2026, 4, 30),
    "claude-sonnet-5-20260401": date(2026, 9, 1),
    "gpt-4o-enterprise-2025-08-deployment": date(2026, 12, 1),
}

WARNING_WINDOW_DAYS = 30


def check_deprecation_status(deployment_config, calendar, today, warning_window_days=WARNING_WINDOW_DAYS):
    retirement_date = calendar.get(deployment_config.pinned_model_version)
    if retirement_date is None:
        return {
            "role": deployment_config.role,
            "pinned_model_version": deployment_config.pinned_model_version,
            "status": "UNKNOWN_NOT_IN_CALENDAR",
            "days_until_retirement": None,
        }

    days_until = (retirement_date - today).days
    if days_until < 0:
        status = "ALREADY_DEPRECATED"
    elif days_until <= warning_window_days:
        status = "DEPRECATION_WARNING"
    else:
        status = "OK"

    return {
        "role": deployment_config.role,
        "pinned_model_version": deployment_config.pinned_model_version,
        "status": status,
        "days_until_retirement": days_until,
    }


print("Deprecation check function defined.")

Deprecation check function defined.


In [8]:
# Run the check "today" = shortly after pinning -- expect a clean OK status.
today_early = date(2026, 3, 22)
early_check = check_deprecation_status(RESEARCH_DEPLOYMENT, MOCK_VENDOR_DEPRECATION_CALENDAR, today_early)
print("Check run shortly after pinning:")
for k, v in early_check.items():
    print(f"  {k}: {v}")
assert early_check["status"] == "OK"
print()
print("PASS: no warning yet, well outside the deprecation window.")

Check run shortly after pinning:
  role: research
  pinned_model_version: claude-sonnet-5-20260315
  status: OK
  days_until_retirement: 39

PASS: no warning yet, well outside the deprecation window.


In [9]:
# Now simulate time passing -- "today" moves to within the warning window of the announced
# retirement date (2026-04-30 in the mock calendar).
today_near_deprecation = date(2026, 4, 15)
late_check = check_deprecation_status(
    RESEARCH_DEPLOYMENT, MOCK_VENDOR_DEPRECATION_CALENDAR, today_near_deprecation
)
print("Check run as the pinned version approaches its announced retirement date:")
for k, v in late_check.items():
    print(f"  {k}: {v}")
assert late_check["status"] == "DEPRECATION_WARNING"
print()
print("PASS: the scheduled check flags the pinned version proactively, BEFORE it stops being "
      "served -- the whole point of running this as a scheduled check rather than discovering "
      "the deprecation when requests start failing.")

Check run as the pinned version approaches its announced retirement date:
  role: research
  pinned_model_version: claude-sonnet-5-20260315
  status: DEPRECATION_WARNING
  days_until_retirement: 15

PASS: the scheduled check flags the pinned version proactively, BEFORE it stops being served -- the whole point of running this as a scheduled check rather than discovering the deprecation when requests start failing.


In [10]:
# And confirm a version that's already past its retirement date is flagged distinctly, too.
today_after_deprecation = date(2026, 5, 5)
expired_check = check_deprecation_status(
    RESEARCH_DEPLOYMENT, MOCK_VENDOR_DEPRECATION_CALENDAR, today_after_deprecation
)
assert expired_check["status"] == "ALREADY_DEPRECATED"
print("Confirmed 'ALREADY_DEPRECATED' status is distinguishable from 'DEPRECATION_WARNING' -- "
      "an operator dashboard built on this check can page differently for an urgent, already-past "
      "deadline versus an upcoming one still inside the planning window.")

Confirmed 'ALREADY_DEPRECATED' status is distinguishable from 'DEPRECATION_WARNING' -- an operator dashboard built on this check can page differently for an urgent, already-past deadline versus an upcoming one still inside the planning window.


## 5. Running the check across every pinned deployment at once

The realistic operational shape: a scheduled job that checks every currently-pinned deployment
config against the vendor deprecation calendar in one pass, and surfaces only the ones that need
attention.

In [11]:
ALL_DEPLOYMENTS = [RESEARCH_DEPLOYMENT, PRODUCTION_BASELINE_DEPLOYMENT]

fleet_check_results = pd.DataFrame(
    [
        check_deprecation_status(cfg, MOCK_VENDOR_DEPRECATION_CALENDAR, today_near_deprecation)
        for cfg in ALL_DEPLOYMENTS
    ]
)
fleet_check_results

,role,pinned_model_version,status,days_until_retirement
0,research,claude-sonnet-5-20260315,DEPRECATION_WARNING,15
1,production_baseline,gpt-4o-enterprise-2025-08-deployment,OK,230


In [12]:
needs_attention = fleet_check_results[
    fleet_check_results["status"].isin(["DEPRECATION_WARNING", "ALREADY_DEPRECATED"])
]
print(f"{len(needs_attention)} of {len(fleet_check_results)} pinned deployments need attention "
      f"as of {today_near_deprecation}:")
needs_attention

1 of 2 pinned deployments need attention as of 2026-04-15:


,role,pinned_model_version,status,days_until_retirement
0,research,claude-sonnet-5-20260315,DEPRECATION_WARNING,15


## Recap

This notebook implemented all three pieces chapter 6 argues are necessary together: an explicit,
dated version pin in the deployment config (never an alias), that same version string stamped on
every logged evaluation result (making a silent mid-research version change auditable rather than
invisible -- section 3), and a scheduled deprecation check that turns a vendor's own announced
retirement date into a proactive warning instead of a surprise outage. None of the three alone closes
the gap chapter 6 describes -- pinning without logging still lets you mix versions in an average
without noticing; logging without a deprecation check still lets a pinned version quietly expire.